In [ ]:
# === COMPLETE END-TO-END EXAMPLE ===
from utils.chunk_processor import ChunkProcessor, ABConfig

# 1. Configure (customize these 3 lines for your use case)
ab_config = ABConfig(
    conversion_event='purchase',           # Your success event
    user_id_col='user_id',                 # Your grouping key
    variant_assignment=lambda uid: np.where(uid % 2 == 0, 'A', 'B')  # Your split logic
)

processor = ChunkProcessor(
    chunk_size=1000000,                    # Rows per chunk
    stats_columns=['event_type', 'price_band'],  # What to validate
    ab_config=ab_config
)

# 2. Process full pipeline
csv_path = 'data/2019-Nov.csv'             # Your giant input
chunks = processor.chunk_csv_to_parquet(csv_path)
validation = processor.validate_chunk_representativeness(chunks)

# 3. Aggregate + global test (single line!)
aggregates = pd.DataFrame([
    processor.aggregate_chunk_ab(chunk) for chunk in chunks
])
results = processor.pool_and_ztest(aggregates)

# 4. Results (production-ready dict)
print(f"Global A/B Test Results:")
print(f"CR A: {results['cr_A']:.3%} ({results['total_conv_A']:,}/{results['total_users_A']:,})")
print(f"CR B: {results['cr_B']:.3%} ({results['total_conv_B']:,}/{results['total_users_B']:,})")
print(f"Relative uplift: {results['relative_uplift_pct']:.1f}%")
print(f"p-value: {results['p_value']:.4f} ({'significant' if results['alpha_005_significant'] else 'not significant'} at α=0.05)")

# Done! results dict ready for charts, dashboards, etc.